In [9]:
model_path1 = f"/home/jupyter/repos/mmelon/biomed-multi-view/model/pgk2_del_cdd_10epochs/last.ckpt"
testpath= f"/home/jupyter/repos/mmelon/biomed-multi-view/ASMS-test/ASMS_PGK2_allNegatives.csv"
#testpath= f"/home/jupyter/repos/mmelon/biomed-multi-view/notebooksdata/inference/temp.csv"
outpath= f"/home/jupyter/repos/mmelon/biomed-multi-view/EmbeddingOutput/asms460k"
out_total=f"/home/jupyter/repos/mmelon/biomed-multi-view/EmbeddingOutput/asms460k/multiview-mmelon.parquet"

In [11]:
import torch
import torch.nn as nn
from bmfm_sm.predictive.data_modules.graph_finetune_dataset import Graph2dFinetuneDataPipeline
from bmfm_sm.predictive.data_modules.image_finetune_dataset import ImageFinetuneDataPipeline
from bmfm_sm.predictive.data_modules.text_finetune_dataset import TextFinetuneDataPipeline
from joblib import Parallel, delayed
from torch.nn.functional import pad
from torch.nn.utils.rnn import pad_sequence
from bmfm_sm.api.smmv_api import SmallMoleculeMultiViewModel, PredictionIterator
from bmfm_sm.core.data_modules.namespace import LateFusionStrategy
from bmfm_sm.api.dataset_registry import DatasetRegistry

In [3]:
class BiomedMultiViewEncoder(nn.Module):
    def __init__(self, model_path=None):
        super(BiomedMultiViewEncoder, self).__init__()
        # Initialize the pretrained model
        biomed_smmv_pretrained = SmallMoleculeMultiViewModel.from_pretrained(
            LateFusionStrategy.ATTENTIONAL, inference_mode=True, model_path=model_path
        )
        # Initialize the model subcomponents
        self.model_graph = biomed_smmv_pretrained.model_graph  # output dim: 512
        self.model_image = biomed_smmv_pretrained.model_image  # output dim: 512
        self.model_text = biomed_smmv_pretrained.model_text  # output dim: 768

    # Helper function for collating the individual processed graph samples:
    @staticmethod
    def collate_graph_data(graph_data_list):
        collated = {}
        collated["node_num"] = torch.cat(
            [sample["node_num"] for sample in graph_data_list]
        )
        collated["node_data"] = torch.cat(
            [sample["node_data"] for sample in graph_data_list]
        )
        collated["edge_num"] = torch.cat(
            [sample["edge_num"] for sample in graph_data_list]
        )
        collated["edge_data"] = torch.cat(
            [sample["edge_data"] for sample in graph_data_list]
        )
        collated["edge_index"] = torch.cat(
            [sample["edge_index"] for sample in graph_data_list], dim=1
        )

        max_node_num = max(collated["node_num"])
        collated["lap_eigvec"] = torch.cat(
            [
                pad(i, (0, max_node_num - i.size(1)), value=float("0"))
                for i in [sample["lap_eigvec"] for sample in graph_data_list]
            ]
        )
        return collated

    def process_smiles(sm):
        img_data = ImageFinetuneDataPipeline.smiles_to_image_format(sm)
        txt_data = TextFinetuneDataPipeline.smiles_to_text_format(sm)
        graph_data = Graph2dFinetuneDataPipeline.smiles_to_graph_format(sm)

        return {
            "img": img_data["img"].squeeze(0),
            "tokens": txt_data["smiles.tokenized"].squeeze(0),
            "attention_mask": txt_data["attention_mask"].squeeze(0),
            "graph": graph_data,
        }

    def forward(self, smiles: list, batch_process: bool = False, n_jobs: int = 1):
        if batch_process:
            results = Parallel(n_jobs=n_jobs)(
                delayed(BiomedMultiViewEncoder.process_smiles)(sm) for sm in smiles
            )
        else:
            results = [BiomedMultiViewEncoder.process_smiles(sm) for sm in smiles]

        image_tensors = [r["img"] for r in results]
        tokenized_smiles_list = [r["tokens"] for r in results]
        attention_mask_list = [r["attention_mask"] for r in results]
        graph_data_list = [r["graph"] for r in results]

        image_batch = torch.stack(image_tensors, dim=0)
        tokenized_smiles_batch = pad_sequence(tokenized_smiles_list, batch_first=True)
        attention_mask_batch = pad_sequence(attention_mask_list, batch_first=True)
        graph_batch = BiomedMultiViewEncoder.collate_graph_data(graph_data_list)

        image_emb = self.model_image(image_batch)
        text_emb = self.model_text(tokenized_smiles_batch, attention_mask_batch)
        graph_emb = self.model_graph(graph_batch)

        return [graph_emb, image_emb, text_emb]

In [4]:
@staticmethod
def process_smiles(sm):
    img_data = ImageFinetuneDataPipeline.smiles_to_image_format(sm)
    txt_data = TextFinetuneDataPipeline.smiles_to_text_format(sm)
    graph_data = Graph2dFinetuneDataPipeline.smiles_to_graph_format(sm)

    return {
        "img": img_data["img"].squeeze(0),
        "tokens": txt_data["smiles.tokenized"].squeeze(0),
        "attention_mask": txt_data["attention_mask"].squeeze(0),
        "graph": graph_data,
    }

In [5]:
# read hugging face model
model = SmallMoleculeMultiViewModel.from_pretrained(LateFusionStrategy.ATTENTIONAL,
                                                    model_path='ibm/biomed.sm.mv-te-84m',
                                                    huggingface=True)

2026-03-19 12:17:24,385 - root - INFO - instance-20260302-162623:140621478803264:0:0 - Loading checkpoint via HuggingFace Hub from provided path ibm/biomed.sm.mv-te-84m
/opt/conda/envs/biomed-multiview/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/envs/biomed-multiview/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
2026-03-19 12:17:25,575 - root - INFO - instance-20260302-162623:140621478803264:0:0 - Using coeff_mlp architecture for aggregator
2026-03-19 12:17:25,576 - root - INFO - instance-20260302-162623:140621478803264:0:0 - dim_list [512, 512, 768] for aggregator
2026-03-19 12:17:25,

In [6]:
# Reading local model
model = SmallMoleculeMultiViewModel.from_pretrained(LateFusionStrategy.ATTENTIONAL,
                                                    model_path=model_path1,
                                                    huggingface=False)

2026-03-19 12:17:26,837 - root - INFO - instance-20260302-162623:140621478803264:0:0 - Using coeff_mlp architecture for aggregator
2026-03-19 12:17:26,838 - root - INFO - instance-20260302-162623:140621478803264:0:0 - dim_list [512, 512, 768] for aggregator
2026-03-19 12:17:26,862 - root - INFO - instance-20260302-162623:140621478803264:0:0 - Loading checkpoint from provided path /home/jupyter/repos/mmelon/biomed-multi-view/model/pgk2_del_cdd_10epochs/last.ckpt
2026-03-19 12:17:27,570 - root - INFO - instance-20260302-162623:140621478803264:0:0 - Loading finetune checkpoint for SmallMoleculeMultiView Model - <All keys matched successfully>
2026-03-19 12:17:27,573 - root - INFO - instance-20260302-162623:140621478803264:0:0 - in train False setting deterministic_eval = True


In [7]:
# Get 512 dimenstion embedding
example_smiles = "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"
example_emb = SmallMoleculeMultiViewModel.get_embeddings(
    smiles=example_smiles,
    model_path="ibm/biomed.sm.mv-te-84m",
    huggingface=True,
)
print(example_emb.shape)

2026-03-19 12:17:27,586 - root - INFO - instance-20260302-162623:140621478803264:0:0 - Loading checkpoint via HuggingFace Hub from provided path ibm/biomed.sm.mv-te-84m
2026-03-19 12:17:28,609 - root - INFO - instance-20260302-162623:140621478803264:0:0 - Using coeff_mlp architecture for aggregator
2026-03-19 12:17:28,610 - root - INFO - instance-20260302-162623:140621478803264:0:0 - dim_list [512, 512, 768] for aggregator
2026-03-19 12:17:28,756 - root - INFO - instance-20260302-162623:140621478803264:0:0 - in train False setting deterministic_eval = True


torch.Size([512])


In [8]:
example_smiles = "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"
example_emb = SmallMoleculeMultiViewModel.get_embeddings(
    smiles=example_smiles,
    model_path= model_path1,
    huggingface=False,
)
print(example_emb.shape)

2026-03-19 12:17:30,223 - root - INFO - instance-20260302-162623:140621478803264:0:0 - Using coeff_mlp architecture for aggregator
2026-03-19 12:17:30,224 - root - INFO - instance-20260302-162623:140621478803264:0:0 - dim_list [512, 512, 768] for aggregator
2026-03-19 12:17:30,242 - root - INFO - instance-20260302-162623:140621478803264:0:0 - Loading checkpoint from provided path /home/jupyter/repos/mmelon/biomed-multi-view/model/pgk2_del_cdd_10epochs/last.ckpt
2026-03-19 12:17:30,649 - root - INFO - instance-20260302-162623:140621478803264:0:0 - Loading finetune checkpoint for SmallMoleculeMultiView Model - <All keys matched successfully>
2026-03-19 12:17:30,652 - root - INFO - instance-20260302-162623:140621478803264:0:0 - in train False setting deterministic_eval = True


torch.Size([512])


In [9]:
model = BiomedMultiViewEncoder(model_path = model_path1)

2026-03-19 12:17:31,816 - root - INFO - instance-20260302-162623:140621478803264:0:0 - Using coeff_mlp architecture for aggregator
2026-03-19 12:17:31,816 - root - INFO - instance-20260302-162623:140621478803264:0:0 - dim_list [512, 512, 768] for aggregator
2026-03-19 12:17:31,835 - root - INFO - instance-20260302-162623:140621478803264:0:0 - Loading checkpoint from provided path /home/jupyter/repos/mmelon/biomed-multi-view/model/pgk2_del_cdd_10epochs/last.ckpt
2026-03-19 12:17:32,242 - root - INFO - instance-20260302-162623:140621478803264:0:0 - Loading finetune checkpoint for SmallMoleculeMultiView Model - <All keys matched successfully>
2026-03-19 12:17:32,244 - root - INFO - instance-20260302-162623:140621478803264:0:0 - in train False setting deterministic_eval = True


In [10]:
import pandas as pd
smiles_list = list(pd.read_csv(testpath)['smiles'])
#print(smils_list[1:5])

In [11]:
# Getting embeddings without batch processing (Faster for smaller lists due to lack of parallelization set-up)
# single_view_embeddings2 = model.forward(smiles_list)

In [12]:
# Getting embeddings with batch processing (Faster for larger lists)
#single_view_embeddings = model.forward(smiles_list, batch_process=True)

In [13]:
# Graph Embeddings, Image Embeddings, Text Embeddings
#single_view_embeddings[0].shape, single_view_embeddings[1].shape, single_view_embeddings[2].shape
#print(single_view_embeddings[0].shape)

In [14]:
'''import pandas as pd
import os
import numpy as np

# Assuming you already have smiles_list from before
# smiles_list = [...]

output_dir = outpath
os.makedirs(output_dir, exist_ok=True)

names_and_tensors = [
    ("graph-mmelon.parquet", single_view_embeddings[0]),
    ("image-mmelon.parquet", single_view_embeddings[1]),
    ("text-mmelon.parquet", single_view_embeddings[2]),
]

for filename, tensor in names_and_tensors:
    arr = tensor.detach().cpu().numpy()  # shape: (N, D)

    # Convert each row to a list (important!)
    embeddings_list = [row.tolist() for row in arr]

    df = pd.DataFrame({
        "smiles": smiles_list,
        "embedding": embeddings_list
    })

    df.to_parquet(os.path.join(output_dir, filename), index=False)

print("Saved all three parquet files with embeddings as arrays.")'''

'import pandas as pd\nimport os\nimport numpy as np\n\n# Assuming you already have smiles_list from before\n# smiles_list = [...]\n\noutput_dir = outpath\nos.makedirs(output_dir, exist_ok=True)\n\nnames_and_tensors = [\n    ("graph-mmelon.parquet", single_view_embeddings[0]),\n    ("image-mmelon.parquet", single_view_embeddings[1]),\n    ("text-mmelon.parquet", single_view_embeddings[2]),\n]\n\nfor filename, tensor in names_and_tensors:\n    arr = tensor.detach().cpu().numpy()  # shape: (N, D)\n\n    # Convert each row to a list (important!)\n    embeddings_list = [row.tolist() for row in arr]\n\n    df = pd.DataFrame({\n        "smiles": smiles_list,\n        "embedding": embeddings_list\n    })\n\n    df.to_parquet(os.path.join(output_dir, filename), index=False)\n\nprint("Saved all three parquet files with embeddings as arrays.")'

In [15]:
import logging
logging.getLogger().setLevel(logging.ERROR)

In [ ]:
import os
import glob
import gc
import numpy as np
import pandas as pd

# =========================================================
# Inputs
# =========================================================
# testpath = "your_input.csv"
# outpath = "your_output_folder"

chunk_size = 1000      # try 200 or 500 first
n_jobs = 4            # safer than very large values like 25

output_dir = outpath
chunk_dir = os.path.join(output_dir, "chunk_outputs")
os.makedirs(chunk_dir, exist_ok=True)

# =========================================================
# Process CSV in chunks
# =========================================================
chunk_idx = 0

for chunk_df in pd.read_csv(testpath, chunksize=chunk_size):
    print(f"Processing chunk {chunk_idx} ...")

    # Keep only smiles column and drop missing values
    chunk_df = chunk_df[["smiles"]].dropna().reset_index(drop=True)

    # Optional: remove empty strings
    chunk_df = chunk_df[chunk_df["smiles"].astype(str).str.strip() != ""].reset_index(drop=True)

    if len(chunk_df) == 0:
        print(f"Chunk {chunk_idx} is empty after cleaning. Skipping.")
        chunk_idx += 1
        continue

    smiles_list = chunk_df["smiles"].tolist()

    # -----------------------------------------------------
    # Compute embeddings for this chunk
    # IMPORTANT:
    # This assumes your model.forward supports n_jobs:
    # model.forward(smiles, batch_process=True, n_jobs=...)
    # -----------------------------------------------------
    single_view_embeddings = model.forward(
        smiles_list,
        batch_process=True,
        n_jobs=n_jobs
    )

    names_and_tensors = [
        (f"graph_chunk_{chunk_idx:04d}.parquet", single_view_embeddings[0]),
        (f"image_chunk_{chunk_idx:04d}.parquet", single_view_embeddings[1]),
        (f"text_chunk_{chunk_idx:04d}.parquet",  single_view_embeddings[2]),
    ]

    # -----------------------------------------------------
    # Save each modality for this chunk
    # -----------------------------------------------------
    for filename, tensor in names_and_tensors:
        arr = tensor.detach().cpu().numpy()   # shape: (N, D)
        embeddings_list = [row.tolist() for row in arr]

        save_df = pd.DataFrame({
            "smiles": smiles_list,
            "embedding": embeddings_list
        })

        save_path = os.path.join(chunk_dir, filename)
        save_df.to_parquet(save_path, index=False)
        print(f"Saved: {save_path}")

        # cleanup
        del arr, embeddings_list, save_df

    # cleanup
    del single_view_embeddings, smiles_list, chunk_df
    gc.collect()

    chunk_idx += 1

print("All chunks processed.")

# =========================================================
# Recombine chunk files into final full parquet files
# =========================================================
final_outputs = {
    "graph": os.path.join(output_dir, "graph-mmelon.parquet"),
    "image": os.path.join(output_dir, "image-mmelon.parquet"),
    "text":  os.path.join(output_dir, "text-mmelon.parquet"),
}

patterns = {
    "graph": os.path.join(chunk_dir, "graph_chunk_*.parquet"),
    "image": os.path.join(chunk_dir, "image_chunk_*.parquet"),
    "text":  os.path.join(chunk_dir, "text_chunk_*.parquet"),
}

for key in ["graph", "image", "text"]:
    files = sorted(glob.glob(patterns[key]))
    if not files:
        print(f"No files found for {key}. Skipping.")
        continue

    print(f"Recombining {len(files)} {key} chunk files ...")

    df_list = [pd.read_parquet(f) for f in files]
    full_df = pd.concat(df_list, ignore_index=True)
    full_df.to_parquet(final_outputs[key], index=False)

    print(f"Saved final {key} file: {final_outputs[key]}")

    del df_list, full_df
    gc.collect()

print("Saved all three final parquet files.")

Processing chunk 0 ...


In [12]:
import logging
logging.getLogger().setLevel(logging.ERROR)

In [ ]:
import os
import glob
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from functools import partial
from embedding_worker import compute_embedding

# =========================================================
# Inputs
# =========================================================
# testpath = "your_input.csv"
# out_total = "your_final_output.parquet"
# model_path1 = "your_model_path"

chunk_size = 200
n_workers = max(1, os.cpu_count() - 1)

output_dir = os.path.dirname(out_total)
chunk_dir = os.path.join(output_dir, "chunk_outputs_total")
os.makedirs(chunk_dir, exist_ok=True)

chunk_idx = 0

for chunk_df in pd.read_csv(testpath, chunksize=chunk_size):
    print(f"Processing chunk {chunk_idx} ...")

    chunk_df = chunk_df[["smiles"]].dropna().reset_index(drop=True)
    chunk_df = chunk_df[
        chunk_df["smiles"].astype(str).str.strip() != ""
    ].reset_index(drop=True)

    if len(chunk_df) == 0:
        print(f"Chunk {chunk_idx} is empty after cleaning. Skipping.")
        chunk_idx += 1
        continue

    smiles_list = chunk_df["smiles"].tolist()

    worker_fn = partial(compute_embedding, model_path1=model_path1)

    results = []
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        for result in tqdm(
            executor.map(worker_fn, smiles_list),
            total=len(smiles_list),
            desc=f"chunk {chunk_idx}"
        ):
            results.append(result)

    smiles_out = [x[0] for x in results]
    all_embeddings = [x[1] for x in results]

    save_df = pd.DataFrame({
        "smiles": smiles_out,
        "embedding": all_embeddings
    })

    chunk_output_path = os.path.join(
        chunk_dir,
        f"embedding_chunk_{chunk_idx:04d}.parquet"
    )
    save_df.to_parquet(chunk_output_path, index=False)
    print(f"Saved: {chunk_output_path}")

    del chunk_df, smiles_list, results, smiles_out, all_embeddings, save_df
    gc.collect()

    chunk_idx += 1

print("All chunks processed.")

embedding_files = sorted(glob.glob(os.path.join(chunk_dir, "embedding_chunk_*.parquet")))

if not embedding_files:
    raise FileNotFoundError(f"No embedding chunk files found in: {chunk_dir}")

print(f"Recombining {len(embedding_files)} embedding chunk files ...")

df_list = [pd.read_parquet(f) for f in embedding_files]
full_df = pd.concat(df_list, ignore_index=True)
full_df.to_parquet(out_total, index=False)

print("Saved final file to:", out_total)
print("Shape:", full_df.shape)

del df_list, full_df
gc.collect()

Processing chunk 0 ...


chunk 0:   0%|          | 0/200 [00:00<?, ?it/s]/opt/conda/envs/biomed-multiview/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/envs/biomed-multiview/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
chunk 0:  84%|████████▍ | 169/200 [03:41<00:40,  1.31s/it]
